#### Simple Gen AI APP Using Langchain

In [19]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [20]:
### data ingestion -> from the website we need to scrape the data 

from langchain_community.document_loaders import WebBaseLoader


In [21]:
loade = WebBaseLoader("https://nsac.basis.org.bd/about-nasa")
loade

In [22]:
loader = loade.load()

In [23]:
loader

[Document(metadata={'source': 'https://nsac.basis.org.bd/about-nasa', 'title': 'NASA Space Apps Challenge', 'description': 'Embark on an extraordinary journey with the NASA International Space Apps Challenge, where innovation meets collaboration, and boundaries are surpassed.', 'language': 'en'}, page_content='\n\n\n\n\nNASA Space Apps Challenge\n\n\n\n\n\n\n\n\n\n\n\n  \n\n\n \n\n\n')]

In [ ]:
### load the data -> Docs -> divide our document into chunks -> texts-> vectors -> vector embedding -----> vectorstoredb

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

documents=text_splitter.split_documents(loader)

In [25]:
documents

[Document(metadata={'source': 'https://nsac.basis.org.bd/about-nasa', 'title': 'NASA Space Apps Challenge', 'description': 'Embark on an extraordinary journey with the NASA International Space Apps Challenge, where innovation meets collaboration, and boundaries are surpassed.', 'language': 'en'}, page_content='NASA Space Apps Challenge')]

In [26]:
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(
    model="embeddinggemma:300m"
)

In [27]:
 from langchain_community.vectorstores import FAISS 

 vectorstoredb = FAISS.from_documents(documents,embeddings)

In [28]:
vectorstoredb

In [29]:
## query from vectorstoredb

query = "Identify Earth Locations that Analog the Permanent Moon Base Locations and Mars"
result = vectorstoredb.similarity_search(query)

result[0].page_content

'NASA Space Apps Challenge'

In [30]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="qwen/qwen3.8-27b"
)

In [31]:
### Retrival Chain , Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based on the provided context:
<context>
{context}
</context>
"""

)
### document chain -> will be responsible in providing my prompt template a specific context 
document_chain = create_stuff_documents_chain(llm,prompt)


In [32]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x000002A5C3B0B230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A5C3B0B770>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [33]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What are 2026 Challenges?",
    "context":[Document(page_content="Choose from 14 challenges for you and your team to address at this year's hackathon! You can use the search and filter features to quickly find a challenge that best matches your interests and skills. 1.Abandoned but not Forgotten: Storytelling about NASA's Discarded Equipment on the Moon and Mars 2.Be An Earth System Trend Detective! ")]
})

"Based on the provided context, you can choose from **14 challenges** for the hackathon. Two specific examples listed are:\n\n1.  **Abandoned but not Forgotten**: Storytelling about NASA's Discarded Equipment on the Moon and Mars.\n2.  **Be An Earth System Trend Detective!**\n\nThe context also notes that you can use search and filter features to find a challenge that best matches your interests and skills."

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [34]:
##### Inout -> Retriver -> vectorstoreDB
### retriver is just an interface, we dont even need to do the similarity search 

vectorstoredb

In [36]:
retriver=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retriver_chain=create_retrieval_chain(retriver,document_chain)

In [37]:
retriver_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A5C390F4D0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
            | ChatGroq(

In [38]:
### get the repossne from the llm 
response=retriver_chain.invoke({"input":"What are 2026 Challenges?"})
response['answer']

'The provided context only contains the title "NASA Space Apps Challenge." Therefore, no specific question can be answered without additional details. Please provide the specific question you would like me to answer based on this context.'

In [39]:
response

{'input': 'What are 2026 Challenges?',
 'context': [Document(id='f6ced01c-15f8-43a8-8635-ac19b832f977', metadata={'source': 'https://nsac.basis.org.bd/about-nasa', 'title': 'NASA Space Apps Challenge', 'description': 'Embark on an extraordinary journey with the NASA International Space Apps Challenge, where innovation meets collaboration, and boundaries are surpassed.', 'language': 'en'}, page_content='NASA Space Apps Challenge')],
 'answer': 'The provided context only contains the title "NASA Space Apps Challenge." Therefore, no specific question can be answered without additional details. Please provide the specific question you would like me to answer based on this context.'}

In [42]:
response['context']

[Document(id='f6ced01c-15f8-43a8-8635-ac19b832f977', metadata={'source': 'https://nsac.basis.org.bd/about-nasa', 'title': 'NASA Space Apps Challenge', 'description': 'Embark on an extraordinary journey with the NASA International Space Apps Challenge, where innovation meets collaboration, and boundaries are surpassed.', 'language': 'en'}, page_content='NASA Space Apps Challenge')]